# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [3]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [4]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Connected to database at: C:\Users\gamal\AppData\Local\Temp\database.sqlite


---

## SQL Exploration

In [5]:
# See all table names
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables:", tables)

Available tables: [('sqlite_sequence',), ('Player_Attributes',), ('Player',), ('Match',), ('League',), ('Country',), ('Team',), ('Team_Attributes',)]


In [6]:
# Load any table into a DataFrame
df = pd.read_sql("SELECT * FROM Country ", connection)
display(df)

,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


In [7]:
df = pd.read_sql("SELECT * FROM League ", connection)
display(df)

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A
5,13274,13274,Netherlands Eredivisie
6,15722,15722,Poland Ekstraklasa
7,17642,17642,Portugal Liga ZON Sagres
8,19694,19694,Scotland Premier League
9,21518,21518,Spain LIGA BBVA


| Columns            	| Beschrijving                                                         	| Relevantie 	|
|--------------------	|----------------------------------------------------------------------	|------------	|
| League(id)         	| Het eigen unieke nummer van de competitie.                           	| 1          	|
| League(country_id) 	| Het nummer dat naar het land verwijst.                               	| 1          	|
| League(name)       	| De naam zoals wij die kennen, bijvoorbeeld "Netherlands Eredivisie". 	| 1          	|
| Country(id)        	| is een uniek codenummer dat de computer gebruikt om te zoeken.       	| 1          	|
| Country(name)      	| De name is gewoon de naam zoals wij die kennen, zoals "Netherlands". 	| 1          	|

### Identificatie en top 10 correlerende factoren voor Sparta Rotterdam

In [ ]:
CLUB_NAAM = 'Sparta Rotterdam'
DOEL_VARIABELE = 'chanceCreationShooting' 

def haal_identifiers_op(connectie, team_naam):
    """
    Haalt de belangrijkste identifiers op voor een specifieke club uit de database.
    """
    # SQL query om identifiers te vinden
    query = f"SELECT team_api_id, team_fifa_api_id, team_short_name FROM Team WHERE team_long_name = '{team_naam}'"
    df = pd.read_sql_query(query, connectie)
    
    # Opslaan in een dictionary
    identifiers = {
        'API_ID': df.iloc[0]['team_api_id'],
        'FIFA_ID': df.iloc[0]['team_fifa_api_id'],
        'Korte_Naam': df.iloc[0]['team_short_name']
    }
    return identifiers


    # Top 10 berekenen
def haal_en_bereid_data_voor(connectie, team_id):
    """
    Haalt data op en zet tekst om naar getallen om meer dan 10 factoren te kunnen analyseren.
    """
    query = f"SELECT * FROM Team_Attributes WHERE team_api_id = {team_id}"
    df = pd.read_sql_query(query, connectie)
    
    # Tekst omzetten naar getallen (0 en 1)
    df_numeriek = pd.get_dummies(df)
    
    return df_numeriek

def bereken_top_10(df, doel):
    """
    Berekent de top 10 factoren met de hoogste correlatie ten opzichte van het doel.
    """
    correlaties = df.corr()[doel].abs()
    
    # Verwijder de ID's en de doelvariabele zelf
    te_verwijderen = [doel, 'id', 'team_fifa_api_id', 'team_api_id']
    correlaties = correlaties.drop(labels=[k for k in te_verwijderen if k in correlaties.index], errors='ignore')
    
    # Sorteer en selecteer de top 10
    top_10 = correlaties.sort_values(ascending=False).head(10)
    return top_10


# Uitvoering van de code

club_identifiers = haal_identifiers_op(connection, CLUB_NAAM)
print("1. Gevonden Identifiers:", club_identifiers)

team_data = haal_en_bereid_data_voor(connection, club_identifiers['API_ID'])

top_10_factoren = bereken_top_10(team_data, DOEL_VARIABELE)

print(f"2. Top 10 correlerende factoren voor '{DOEL_VARIABELE}':")

print(top_10_factoren)

1. Gevonden Identifiers: {'API_ID': np.int64(8614), 'FIFA_ID': np.int64(100646), 'Korte_Naam': 'SPA'}
2. Top 10 correlerende factoren voor 'chanceCreationShooting':
buildUpPlaySpeed                   NaN
buildUpPlayPassing                 NaN
chanceCreationPassing              NaN
chanceCreationCrossing             NaN
defencePressure                    NaN
defenceAggression                  NaN
defenceTeamWidth                   NaN
date_2010-02-22 00:00:00           NaN
buildUpPlaySpeedClass_Balanced     NaN
buildUpPlayDribblingClass_Little   NaN
Name: chanceCreationShooting, dtype: float64
